In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [3]:
train_generator = datagen.flow_from_directory(
    "data/train",
    target_size=(64, 64),
    color_mode="grayscale",
    batch_size=32,
    class_mode="binary",
    subset="training"
)

Found 67919 images belonging to 2 classes.


In [4]:
val_generator = datagen.flow_from_directory(
    "data/train",
    target_size=(64, 64),
    color_mode="grayscale",
    batch_size=32,
    class_mode="binary",
    subset="validation"
)

Found 16979 images belonging to 2 classes.


In [5]:
from tensorflow.keras import layers, models

model = models.Sequential()

# First Conv Block
model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,1)))
model.add(layers.MaxPooling2D(2,2))

# Second Conv Block
model.add(layers.Conv2D(64, (3,3), activation='relu'))
model.add(layers.MaxPooling2D(2,2))

# Flatten
model.add(layers.Flatten())

# Dense Layer
model.add(layers.Dense(128, activation='relu'))

# Dropout
model.add(layers.Dropout(0.5))

# Output Layer
model.add(layers.Dense(1, activation='sigmoid'))


d:\PROJECTS\DRIVER_DROSSINESS_SYSTEM\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)


Epoch 1/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 169s 79ms/step - accuracy: 0.9317 - loss: 0.1811 - val_accuracy: 0.8667 - val_loss: 0.3771
Epoch 2/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 139s 66ms/step - accuracy: 0.9657 - loss: 0.0990 - val_accuracy: 0.9016 - val_loss: 0.2886
Epoch 3/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 149s 70ms/step - accuracy: 0.9738 - loss: 0.0777 - val_accuracy: 0.8756 - val_loss: 0.3502
Epoch 4/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 139s 65ms/step - accuracy: 0.9779 - loss: 0.0645 - val_accuracy: 0.8731 - val_loss: 0.4652
Epoch 5/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 139s 65ms/step - accuracy: 0.9809 - loss: 0.0555 - val_accuracy: 0.8859 - val_loss: 0.3988
Epoch 6/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 139s 65ms/step - accuracy: 0.9830 - loss: 0.0502 - val_accuracy: 0.8809 - val_loss: 0.3935
Epoch 7/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 139s 65ms/step - accuracy: 0.9842 - loss: 0.0457 - val_accuracy: 0.8789 - val_loss: 0.4267
Epoch 8/10
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 143s 67ms/step - accuracy: 

In [8]:
model.save("eye_model.keras")

In [9]:
import cv2

# Load Haar Cascades
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_eye.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray)

        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey),
                          (ex+ew, ey+eh), (0, 255, 0), 2)

    cv2.imshow("Face & Eye Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 